<p align="left">
  <img src="https://miro.medium.com/v2/resize:fit:1358/1*bSLNlG7crv-p-m4LVYYk3Q.png" width="360" style="display:inline-block;"/>
</p>

# Detecção de Bandeiras de Países com YOLOv8

Este notebook realiza a detecção de **86 bandeiras nacionais** usando YOLOv8s.

Fluxo completo:
1. Upload do dataset limpo (formato YOLOv8) direto no Colab
2. Treinamento de 60 épocas com `yolov8s.pt`
3. Avaliação de métricas (mAP@0.5, Precisão, Recall)
4. Predição em **fotos reais capturadas pelo grupo**

# Pacotes

Instala o pacote `ultralytics` (inclui YOLOv8) e verifica o ambiente Colab.

In [ ]:
# Instala o Ultralytics (inclui YOLOv8) — normalmente já disponível no Colab
!pip install ultralytics --quiet

import ultralytics
ultralytics.checks()   # exibe versão, GPU disponível, etc.

In [ ]:
import os
import glob
import random
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

# Sobre o Dataset: Country Flags Detection

Dataset obtido via [Roboflow Universe](https://universe.roboflow.com/phamdata/country-flags-2t33e/dataset/4).

## Divisao dos dados (apos limpeza):
- **Total de imagens:** 10.187
  - Treino : 8.940 imagens
  - Validacao: 827 imagens
  - Teste : 420 imagens

## Classes detectadas: **86 bandeiras nacionais**
De Algeria ate Zambia (Afghanistan e Albania foram removidas por anotacoes invalidas).

## Licenca: CC BY 4.0

## Upload do Dataset no Colab

Faca o upload do arquivo **`dataset_clean.zip`** quando solicitado.

> O dataset limpo ja vem com `data.yaml` corrigido e classes reindexadas (0..85).

In [ ]:
from google.colab import files
import zipfile

# Aguarda o usuario selecionar o arquivo dataset_clean.zip
print("Selecione o arquivo dataset_clean.zip para upload...")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
print(f"Arquivo recebido: {zip_name}")

# Extrai para /content/dataset
os.makedirs('/content/dataset', exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/dataset')
print("Dataset extraido em /content/dataset/")

# Contagem rapida por split
for split in ['train', 'valid', 'test']:
    img_dir = f'/content/dataset/{split}/images'
    lbl_dir = f'/content/dataset/{split}/labels'
    if os.path.isdir(img_dir):
        n_imgs = len([f for f in os.listdir(img_dir)
                      if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
        n_lbls = len([f for f in os.listdir(lbl_dir) if f.endswith('.txt')])
        print(f"  {split:6s}: {n_imgs:5d} imagens | {n_lbls:5d} labels")

## Configuracao do `data.yaml`

A celula abaixo reescreve o `data.yaml` com caminhos absolutos do Colab,
garantindo que o treinamento encontre as imagens corretamente.

In [ ]:
import yaml

# Caminhos absolutos para o ambiente Colab
data_yaml = {
    'train': '/content/dataset/train/images',
    'val'  : '/content/dataset/valid/images',
    'test' : '/content/dataset/test/images',
    'nc'   : 86,
    'names': ["Algeria Flag", "American Samoa", "Andorra Flag", "Angola Flag", "Anguilla Flag", "Antigua and Barbuda", "Argentina", "Armenia", "Australia", "Austria", "Bahrain", "Belgium", "Bolivia", "Bosnia and Herzegovina", "Brazil", "Cameroon", "Canada", "Cape Verde", "Chile", "China", "Colombia", "Costa Rica", "Croatia", "Czech Republic", "Democratic Republic of the Congo", "Denmark", "Ecuador", "Egypt", "El Salvador", "England", "Finland", "France", "Gabon", "Georgia", "Germany", "Ghana", "Greece", "Haiti", "Honduras", "Hungary", "Iceland", "Iran", "Iraq", "Israel", "Italy", "Ivory Coast", "Jamaica", "Japan", "Jordan", "Madagascar", "Mali", "Mexico", "Morocco", "Netherlands", "New Zealand", "Nigeria", "Norway", "Panama", "Paraguay", "Peru", "Poland", "Portugal", "Romania", "Saudi Arabia", "Scotland", "Senegal", "Serbia", "Slovakia", "Slovenia", "South Africa", "South Korea", "Spain", "Sweden", "Switzerland", "Tanzania", "Tunisia", "Turkey", "Ukraine", "United Arab Emirates", "United States", "Uruguay", "Uzbekistan", "Venezuela", "Vietnam", "Wales", "Zambia"],
}

yaml_path = '/content/dataset/data.yaml'
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(data_yaml, f, sort_keys=False, allow_unicode=True)

print("data.yaml configurado:")
with open(yaml_path) as f:
    print(f.read()[:400], "...")  # mostra o inicio

In [ ]:
# Lista de nomes das 86 classes (mesma ordem do data.yaml)
nomes_classes = ['Algeria Flag', 'American Samoa', 'Andorra Flag', 'Angola Flag', 'Anguilla Flag', 'Antigua and Barbuda', 'Argentina', 'Armenia', 'Australia', 'Austria', 'Bahrain', 'Belgium', 'Bolivia', 'Bosnia and Herzegovina', 'Brazil', 'Cameroon', 'Canada', 'Cape Verde', 'Chile', 'China', 'Colombia', 'Costa Rica', 'Croatia', 'Czech Republic', 'Democratic Republic of the Congo', 'Denmark', 'Ecuador', 'Egypt', 'El Salvador', 'England', 'Finland', 'France', 'Gabon', 'Georgia', 'Germany', 'Ghana', 'Greece', 'Haiti', 'Honduras', 'Hungary', 'Iceland', 'Iran', 'Iraq', 'Israel', 'Italy', 'Ivory Coast', 'Jamaica', 'Japan', 'Jordan', 'Madagascar', 'Mali', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'Nigeria', 'Norway', 'Panama', 'Paraguay', 'Peru', 'Poland', 'Portugal', 'Romania', 'Saudi Arabia', 'Scotland', 'Senegal', 'Serbia', 'Slovakia', 'Slovenia', 'South Africa', 'South Korea', 'Spain', 'Sweden', 'Switzerland', 'Tanzania', 'Tunisia', 'Turkey', 'Ukraine', 'United Arab Emirates', 'United States', 'Uruguay', 'Uzbekistan', 'Venezuela', 'Vietnam', 'Wales', 'Zambia']

print(f"Total de classes: {len(nomes_classes)}")
print("Primeiras 5:", nomes_classes[:5])
print("Ultimas  5:", nomes_classes[-5:])

## Visualizacao de Amostras (Test Set)

Exibe 4 imagens aleatorias do conjunto de teste com suas bounding boxes.

In [ ]:
# Diretórios do conjunto de teste
test_img_dir = '/content/dataset/test/images'
test_lbl_dir = '/content/dataset/test/labels'

# Seleciona 4 imagens aleatorias
image_files = [f for f in os.listdir(test_img_dir)
               if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
sample_images = random.sample(image_files, min(4, len(image_files)))

plt.figure(figsize=(16, 4))

for i, img_file in enumerate(sample_images):
    img_path = os.path.join(test_img_dir, img_file)
    lbl_path = os.path.join(test_lbl_dir, os.path.splitext(img_file)[0] + '.txt')

    img = Image.open(img_path).convert('RGB')
    draw = ImageDraw.Draw(img)
    w, h = img.size
    font_size = max(int(w * 0.06), 12)

    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                cls_id = int(parts[0])
                xc, yc, bw, bh = map(float, parts[1:5])
                x1 = (xc - bw / 2) * w
                y1 = (yc - bh / 2) * h
                x2 = (xc + bw / 2) * w
                y2 = (yc + bh / 2) * h
                draw.rectangle([x1, y1, x2, y2], outline='red', width=3)
                nome = nomes_classes[cls_id] if cls_id < len(nomes_classes) else f'Classe {cls_id}'
                draw.text((x1, max(y1 - font_size, 0)), nome, fill='red', font_size=font_size)

    plt.subplot(1, 4, i + 1)
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'Amostra {i + 1}')

plt.suptitle('Exemplos do conjunto de teste com bounding boxes', fontsize=13)
plt.tight_layout()
plt.show()

## Estrutura esperada do dataset (YOLOv8)

```
dataset/
├── train/
│   ├── images/   ← imagens de treino
│   └── labels/   ← arquivos .txt com bounding boxes
├── valid/
│   ├── images/
│   └── labels/
├── test/
│   ├── images/
│   └── labels/
└── data.yaml     ← configuracao do dataset
```

# Treinamento do Modelo YOLO

## Escolha do Modelo

Usamos **YOLOv8s** (small) — bom equilibrio entre velocidade e precisao para 86 classes.

| Variante | Parametros | GFLOPs | Notas                          |
|----------|-----------|--------|-------------------------------|
| n (nano) | 3.2 M     | 8.7    | Muito rapido, menos preciso   |
| **s (small)** | **11.2 M** | **28.6** | **Nosso padrao** |
| m (medium) | 25.9 M  | 78.9   | Mais preciso, 2x mais lento   |
| l (large) | 43.7 M   | 165    | Alta precisao, GPU potente    |
| x (extra) | 68.2 M   | 258    | Maximo desempenho             |

Documentacao completa de parametros: https://docs.ultralytics.com/pt/modes/train/

In [ ]:
import gc
import torch
from ultralytics import YOLO

# Libera memoria da GPU antes de comecar
gc.collect()
torch.cuda.empty_cache()

# Carrega o modelo pre-treinado YOLOv8s
# Na primeira execucao o Colab baixa automaticamente os pesos do COCO
model = YOLO('yolov8s.pt')

# Inicia o treinamento
# - epochs=60: suficiente para convergir com 86 classes
# - imgsz=640: resolucao padrao YOLOv8 (boa para detalhes de bandeiras)
# - batch=16 : cabe na GPU T4 do Colab (16 GB VRAM)
#              se der OOM, reduza para 8 ou use batch=-1 (autobatch)
# - patience=15: para antecipadamente se nao houver melhora por 15 epocas
model.train(
    data='/content/dataset/data.yaml',
    epochs=60,
    imgsz=640,
    batch=16,
    patience=15,
    project='runs/detect',
    name='bandeiras',
    exist_ok=True,
)

> **Nota:** Nao e preciso recarregar o modelo se voce acabou de treinar.
> A variavel `model` ja aponta para os melhores pesos.
> O bloco abaixo so e necessario se voce reabrir o notebook apos o treino.

In [ ]:
# Descomente para carregar o modelo ja treinado (util ao reabrir o notebook)
# model = YOLO('runs/detect/bandeiras/weights/best.pt')
# print("Modelo carregado:", model.info())

## Visualizacao do Batch de Treino (Data Augmentation)

O YOLO aplica augmentations automaticamente durante o treino
(flips, HSV jitter, mosaico, etc.).
A imagem abaixo mostra um batch real com essas transformacoes.

In [ ]:
# Detecta a pasta do ultimo treino (evita erros com train2, train3...)
try:
    aug_dir = model.trainer.save_dir
except AttributeError:
    aug_dir = 'runs/detect/bandeiras'

aug_img_path = os.path.join(aug_dir, 'train_batch0.jpg')

if os.path.exists(aug_img_path):
    img = Image.open(aug_img_path)
    plt.figure(figsize=(12, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Exemplo de Batch de Treino com Data Augmentation')
    plt.show()
else:
    print(f"Imagem nao encontrada em '{aug_img_path}'.")
    print("Isso e normal se voce pulou o treinamento e carregou best.pt diretamente.")

# Avaliacao do Modelo

In [ ]:
# Avalia o modelo no conjunto de TESTE (nao visto durante o treino)
results = model.val(data='/content/dataset/data.yaml', split='test')

In [ ]:
# Exibe as principais metricas de desempenho
metrics = results.results_dict
print("=" * 45)
print(f"  Precisao media  : {metrics.get('metrics/precision(B)', 0):.4f}")
print(f"  Recall medio    : {metrics.get('metrics/recall(B)', 0):.4f}")
print(f"  mAP@0.5         : {metrics.get('metrics/mAP50(B)', 0):.4f}")
print(f"  mAP@0.5:0.95    : {metrics.get('metrics/mAP50-95(B)', 0):.4f}")
print("=" * 45)

## Graficos de Treinamento e Matriz de Confusao

O YOLO salva automaticamente os graficos de treino e a matriz de confusao
na pasta do treinamento. As celulas abaixo exibem essas imagens
(use-as no relatorio tecnico).

In [ ]:
# Pasta onde o YOLO salvou os artefatos do treino
try:
    run_dir = model.trainer.save_dir
except AttributeError:
    run_dir = 'runs/detect/bandeiras'

# Lista de artefatos uteis para o relatorio
artefatos = {
    'results.png'                  : 'Curvas de Treino (loss, mAP, precision, recall)',
    'confusion_matrix_normalized.png': 'Matriz de Confusao (normalizada)',
    'BoxPR_curve.png'              : 'Curva Precision x Recall',
    'BoxF1_curve.png'              : 'Curva F1 x Confianca',
}

for arquivo, titulo in artefatos.items():
    caminho = os.path.join(run_dir, arquivo)
    if os.path.exists(caminho):
        img = Image.open(caminho)
        plt.figure(figsize=(14, 8))
        plt.imshow(img)
        plt.axis('off')
        plt.title(titulo, fontsize=14)
        plt.show()
    else:
        print(f"Nao encontrado: {caminho}")

## Inferencia no Conjunto de Teste

Alem das fotos do grupo, o enunciado pede inferencias nas imagens do
conjunto de teste. Aqui o modelo PREVE (nao usa o gabarito) em 4 imagens
aleatorias do test set.

In [ ]:
# Seleciona 4 imagens aleatorias do conjunto de teste
test_imgs = glob.glob('/content/dataset/test/images/*.jpg')
amostra = random.sample(test_imgs, min(4, len(test_imgs)))

# Predicao do modelo (sem olhar os labels reais)
preds_test = [model.predict(p, conf=0.25, verbose=False)[0] for p in amostra]

plt.figure(figsize=(16, 4))
for i, pred in enumerate(preds_test):
    img_rgb = pred.plot()[:, :, ::-1]   # BGR -> RGB
    plt.subplot(1, 4, i + 1)
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title(f'Teste {i + 1}')
plt.suptitle('Predicoes do modelo no conjunto de teste', fontsize=13)
plt.tight_layout()
plt.show()

# Predicao nas Fotos Capturadas pelo Grupo

Aqui voce fara upload das **fotos reais de bandeiras** tiradas pelo grupo.

> Requisito do enunciado: as imagens devem ser **capturadas pelo proprio grupo**,
> nao baixadas da internet.

## Upload das Fotos Reais

Clique em "Escolher arquivos" e selecione as fotos de bandeiras capturadas pelo grupo.
Formatos aceitos: `.jpg`, `.jpeg`, `.png`

In [ ]:
from google.colab import files

# Upload das fotos reais de bandeiras capturadas pelo grupo
print("Envie as fotos de bandeiras capturadas pelo grupo:")
uploaded_fotos = files.upload()

# Lista de caminhos das imagens enviadas
image_paths = list(uploaded_fotos.keys())
print(f"\n{len(image_paths)} imagem(ns) carregada(s):")
for p in image_paths:
    print(f"  - {p}")

## Resultado das Deteccoes

In [ ]:
# Realiza a predicao em todas as imagens enviadas
# conf=0.25: confianca minima para exibir uma deteccao
preds = [model.predict(img_path, conf=0.25, verbose=False)[0] for img_path in image_paths]

# Plota as imagens com bounding boxes desenhadas pelo proprio YOLO
n = len(preds)
plt.figure(figsize=(7 * n, 6))

for i, pred in enumerate(preds):
    # .plot() retorna array BGR com caixas, labels e confiancas sobrepostos
    img_bgr = pred.plot()
    img_rgb = img_bgr[:, :, ::-1]   # BGR -> RGB para o matplotlib

    plt.subplot(1, n, i + 1)
    plt.imshow(img_rgb)
    plt.title(f'Foto {i + 1}', fontsize=13)
    plt.axis('off')

plt.suptitle('Deteccao de Bandeiras nas Fotos do Grupo', fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
# Exibe os detalhes textuais de cada deteccao (classe, confianca, bbox)
for i, (pred, path) in enumerate(zip(preds, image_paths)):
    print(f"\nFoto {i + 1}: {path}")
    if len(pred.boxes) == 0:
        print("  Nenhuma bandeira detectada.")
        print("  Dica: diminua conf=0.25 para conf=0.10 na celula acima e execute de novo.")
        continue
    for box in pred.boxes:
        cls_id = int(box.cls[0])
        conf   = float(box.conf[0])
        nome   = nomes_classes[cls_id] if cls_id < len(nomes_classes) else f'Classe {cls_id}'
        # Coordenadas absolutas da bbox
        x1, y1, x2, y2 = [round(v, 1) for v in box.xyxy[0].tolist()]
        print(f"  [{conf:.1%}] {nome}  bbox=({x1},{y1},{x2},{y2})")